<a href="https://www.kaggle.com/code/rawanmoamed/eeg-classification-ccnn?scriptVersionId=295616260" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install torch_scatter torcheeg torch_geometric -qq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.1/295.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 62.7 MB/s eta 0:00:00

In [2]:
import torch
import torch.nn as nn
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Subset,WeightedRandomSampler
from torcheeg.models import CCNN
from torcheeg import transforms
from torcheeg.transforms import ToGrid
from torcheeg.datasets import SEEDIVDataset,SEEDIVFeatureDataset
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
from torcheeg.transforms import ToG
from torcheeg.datasets.constants import SEED_IV_ADJACENCY_MATRIX
from torcheeg.models import DGCNN
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.lr_scheduler import ReduceLROnPlateau 
# --- THE MAIN SUBJECT LOOP ---

import torch_geometric.loader as geom_loader # Special loader for graphs
import copy
import scipy.signal as signal
import random
import numpy as np

In [3]:
import shutil
import os

if os.path.exists('./tmp_out/seed_iv_features'):
    shutil.rmtree('./tmp_out/seed_iv_features')
    print("Old cache deleted. Data will be re-processed.")

In [4]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42) 

In [6]:
def weights_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

In [7]:
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
from torcheeg.transforms import ToGrid



dataset = SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'],

    offline_transform=transforms.Compose([
        ToGrid(SEED_IV_CHANNEL_LOCATION_DICT),
        transforms.MinMaxNormalize(),
        transforms.Lambda(lambda x: torch.tensor(x).float())
    ]),

    label_transform=transforms.Compose([
        transforms.Select('emotion')
    ]),

    io_mode='memory'
)



[2026-02-03 08:26:21] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features.
[2026-02-03 08:26:21] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 121it [00:00, 1209.81it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 657it [00:00, 3647.06it/s]
[PROCESS]: 100%|██████████| 45/45 [00:10<00:00,  4.36it/s]


In [8]:
subjects = sorted(dataset.info['subject_id'].unique()) # Get list of all 15 subjects


In [9]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        x = batch[0].to(device)
        y = batch[1].to(device)


        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader), correct / total


In [10]:
def evaluate(model, loader, criterion):
    model.eval()
    correct, total = 0, 0
    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()
            preds = out.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / len(loader), correct / total


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ImprovedCCNN(nn.Module):
    def __init__(self, in_channels=5, num_classes=4, grid_size=(9,9), dropout=0.5):
        super(ImprovedCCNN, self).__init__()
        self.grid_h, self.grid_w = grid_size

        # --- Convolutional Layers ---
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(dropout)

        # --- Global Average Pooling ---
        self.gap = nn.AdaptiveAvgPool2d((1,1))

        # --- Fully Connected Layers ---
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # Convolution + BN + GELU
        x = F.gelu(self.bn1(self.conv1(x)))
        x = F.gelu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = F.gelu(self.bn3(self.conv3(x)))
        x = self.pool3(x)

        x = self.dropout(x)
        x = self.gap(x)             # (B, 128, 1, 1)
        x = x.view(x.size(0), -1)   # flatten to (B, 128)

        # Fully Connected
        x = F.gelu(self.fc1(x))
        x = self.fc2(x)
        return x


In [12]:
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

In [13]:
all_subject_accuracies = []
epochs=100
for subject_id in subjects:
    print(f"\nProcessing SUBJECT: {subject_id}")
    
    # --- Data Split by Trials ---
    sub_df = dataset.info[dataset.info['subject_id'] == subject_id]
    all_trials = list(range(1, 25))
    random.seed(42)
    test_trials = random.sample(all_trials, 5)
    train_trials = [t for t in all_trials if t not in test_trials]

    train_indices = sub_df[sub_df['trial_id'].isin(train_trials)].index.tolist()
    test_indices = sub_df[sub_df['trial_id'].isin(test_trials)].index.tolist()

    train_set = Subset(dataset, train_indices)
    test_set = Subset(dataset, test_indices)

    # --- Sampler for Class Imbalance ---
    y_train_indices = train_set.indices
    raw_labels = dataset.info.iloc[y_train_indices]['emotion'].values
    class_counts = np.bincount(raw_labels)
    class_weights = 1. / class_counts
    sample_weights = [class_weights[y] for y in raw_labels]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_set, batch_size=64, sampler=sampler, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)

    # --- Model ---
    model = ImprovedCCNN(in_channels=5, num_classes=4, dropout=0.5).to(device)

    # --- Criterion & Optimizer ---
    criterion = nn.CrossEntropyLoss(label_smoothing=0.15).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # --- Scheduler: OneCycleLR ---
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.01,
        steps_per_epoch=len(train_loader),
        epochs=epochs
    )

    best_acc = 0.0

    # --- Training Loop ---
    for epoch in range(epochs):
        model.train()
        train_loss_sum = 0
        correct = 0
        total = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss_sum += loss.item()
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        avg_train_loss = train_loss_sum / len(train_loader)
        train_acc = correct / total

        val_loss, val_acc = evaluate(model, test_loader, criterion)

        # Save best model
        is_best = False
        if val_acc > best_acc:
            best_acc = val_acc
            is_best = True
            save_path = os.path.join(save_dir, f"best_model_subject_{subject_id}.pth")
            torch.save(model.state_dict(), save_path)

        if is_best or (epoch+1) % 5 == 0:
            status = "(*) BEST" if is_best else ""
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.3f} | "
                  f"Val Acc: {val_acc:.3f} {status}")

    all_subject_accuracies.append(best_acc)
    print(f"Subject {subject_id} FINAL BEST ACC: {best_acc*100:.2f}%")

# --- Final Summary ---
print("\n" + "="*50)
print("FINAL RESULTS SUMMARY")
print("="*50)
final_avg_acc = np.mean(all_subject_accuracies)
final_std_acc = np.std(all_subject_accuracies)
print(f"Subjects: {len(subjects)} | Average Accuracy: {final_avg_acc*100:.2f}% | Std: {final_std_acc*100:.2f}")
print(f"Accuracies List: {[round(x*100,2) for x in all_subject_accuracies]}")


Processing SUBJECT: 1
Epoch 1/100 | Train Loss: 1.3823 | Val Loss: 1.3892 | Train Acc: 0.287 | Val Acc: 0.320 (*) BEST
Epoch 5/100 | Train Loss: 1.1208 | Val Loss: 4.2830 | Train Acc: 0.553 | Val Acc: 0.125 
Epoch 7/100 | Train Loss: 0.8245 | Val Loss: 1.3117 | Train Acc: 0.799 | Val Acc: 0.528 (*) BEST
Epoch 10/100 | Train Loss: 0.5739 | Val Loss: 4.5330 | Train Acc: 0.973 | Val Acc: 0.125 
Epoch 15/100 | Train Loss: 0.5130 | Val Loss: 1.6719 | Train Acc: 0.999 | Val Acc: 0.467 
Epoch 20/100 | Train Loss: 0.6444 | Val Loss: 8.2220 | Train Acc: 0.914 | Val Acc: 0.125 
Epoch 25/100 | Train Loss: 0.4896 | Val Loss: 4.0165 | Train Acc: 0.999 | Val Acc: 0.125 
Epoch 26/100 | Train Loss: 0.4883 | Val Loss: 1.1512 | Train Acc: 1.000 | Val Acc: 0.640 (*) BEST
Epoch 30/100 | Train Loss: 0.4818 | Val Loss: 1.0482 | Train Acc: 1.000 | Val Acc: 0.743 (*) BEST
Epoch 35/100 | Train Loss: 0.4928 | Val Loss: 1.5320 | Train Acc: 1.000 | Val Acc: 0.287 
Epoch 40/100 | Train Loss: 0.4831 | Val Loss: 1.